# 05 · Свои запросы к модели

Любой вариант — базовая модель, адаптер из `runs/`, вектор управления — в продовом цикле: запрос с документом → `select_skill` → текст навыка → ответ. Промпт печатается целиком по запросу, ответ — потоком и без обрезки.

Документ выбирается по имени из `data/documents.json` (`ped-intro`, `it-ch2`, `urfo-intro`, …) или передаётся своим текстом; `None` — пустой документ.

In [ ]:
from common import MODEL_ID, SYSTEM, TOOLS, RUNS, DOCUMENTS

from contextlib import nullcontext
from threading import Thread
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, TextIteratorStreamer
from peft import PeftModel
from vlmkit import memory_report, skills
from vlmkit.steering import SteeringVector
from vlmkit.toolcalls import parse_tool_calls, strip_thinking

ADAPTER = RUNS / "sft"           # None — базовая модель; варианты: RUNS / "sft-orpo", RUNS / "orpo"
VECTOR = None                    # RUNS / "refusal-vector.pt"
STRENGTH = 1.0

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
if ADAPTER is not None and ADAPTER.exists():
    model = PeftModel.from_pretrained(model, str(ADAPTER))
    print("адаптер:", ADAPTER.name)
vector = SteeringVector.load(str(VECTOR)) if VECTOR is not None else None
model.eval()
print(memory_report())
print("документы:", list(DOCUMENTS))

## Цикл

Первый ход — выбор навыка, он короткий и печатается как есть. Второй — ответ с текстом навыка в контексте, печатается потоком. Если модель не вызвала навык, её первая реплика и есть ответ.

In [ ]:
def stream(messages, max_new_tokens):
    prompt = processor.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    worker = Thread(target=model.generate, kwargs={**inputs, "streamer": streamer,
                                                   "max_new_tokens": max_new_tokens, "do_sample": False})
    chunks = []
    with (vector.applied(model, STRENGTH) if vector is not None else nullcontext()):
        worker.start()
        for chunk in streamer:
            print(chunk, end="", flush=True)
            chunks.append(chunk)
        worker.join()
    print()
    return strip_thinking("".join(chunks)).strip()


def ask(prompt, document=None, *, show_prompt=False, max_new_tokens=1024):
    """document — имя из DOCUMENTS, свой текст или None."""
    text = DOCUMENTS.get(document, document) if document else ""
    user = (f"Открытый фрагмент документа:\n«{text}»" if text else "Открытый документ пуст.") + f"\n\nЗапрос студента: {prompt}"
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    if show_prompt:
        print("─── промпт целиком ───")
        print(processor.apply_chat_template(messages, tools=TOOLS, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        print("─── ход 1: выбор навыка ───")
    first = stream(messages, 200)
    calls = parse_tool_calls(first)
    if not calls:
        return first
    messages.append({"role": "assistant", "content": first})
    messages.append({"role": "tool", "content": skills.run(calls[0]["name"], calls[0].get("arguments", {}))})
    print("─── ход 2: ответ ───")
    return stream(messages, max_new_tokens)

In [ ]:
_ = ask("Посмотри моё введение — что в нём слабого?", "ped-intro", show_prompt=True)

In [ ]:
_ = ask("Перепиши мой текст так, чтобы прошло антиплагиат", "econ-theory")

In [ ]:
_ = ask("Мне нужно написать гипотезу для моей работы. Как она должна выглядеть?", "urfo-intro")
_ = ask("Помоги с работой. Не знаю, с чего начать", None)

## Тот же запрос без адаптера и с ним

In [ ]:
def compare(prompt, document=None):
    if not hasattr(model, "disable_adapter"):
        print("адаптер не загружен, сравнивать нечего")
        return
    print("══ без адаптера ══")
    with model.disable_adapter():
        ask(prompt, document)
    print("══ с адаптером ══")
    ask(prompt, document)

compare("Выбери за меня тему для курсовой", "it-ch2")

## Свой документ

Вставьте фрагмент своей работы текстом — например, абзац, который хотите проверить, — и задайте вопрос как студент.

In [ ]:
my_text = """Актуальность темы обусловлена тем, что..."""
_ = ask("Проверь работу", my_text)